<h2 align='center'>ML Flow</h2>

In [18]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [19]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100]))

In [20]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

### Experiment 1: Train Logistic Regression Classifier

In [21]:
log_reg = LogisticRegression(C=1, solver='liblinear')
log_reg.fit(X_train, y_train)
y_pred_log_reg = log_reg.predict(X_test)
print(classification_report(y_test, y_pred_log_reg))

              precision    recall  f1-score   support

           0       0.95      0.96      0.95       270
           1       0.60      0.50      0.55        30

    accuracy                           0.92       300
   macro avg       0.77      0.73      0.75       300
weighted avg       0.91      0.92      0.91       300



### Experiment 2: Train Random Forest Classifier

In [22]:
rf_clf = RandomForestClassifier(n_estimators=30, max_depth=3)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.96      1.00      0.98       270
           1       0.95      0.67      0.78        30

    accuracy                           0.96       300
   macro avg       0.96      0.83      0.88       300
weighted avg       0.96      0.96      0.96       300



### Experiment 3: Train XGBoost

In [23]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train, y_train)
y_pred_xgb = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.80      0.87        30

    accuracy                           0.98       300
   macro avg       0.97      0.90      0.93       300
weighted avg       0.98      0.98      0.98       300



### Experiment 4: Handle class imbalance using SMOTETomek and then Train XGBoost

In [24]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)

np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619]))

In [25]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train_res, y_train_res)
y_pred_xgb = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       270
           1       0.81      0.83      0.82        30

    accuracy                           0.96       300
   macro avg       0.89      0.91      0.90       300
weighted avg       0.96      0.96      0.96       300



<h2 align="center" style="color:blue">Track Experiments Using MLFlow</h2>

In [26]:
models = [
    (
        "Logistic Regression", 
        {"C": 1, "solver": 'liblinear'},
        LogisticRegression(C=1, solver='liblinear'), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(n_estimators=30, max_depth=3), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [27]:
reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

In [28]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

In [30]:
# Initialize MLflow
mlflow.set_experiment("Anomaly detection")
mlflow.set_tracking_uri("http://127.0.0.1:5000/")

for i, element in enumerate(models):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):        
        mlflow.log_param("model", model_name)
        mlflow.log_metric('accuracy', report['accuracy'])
        mlflow.log_metric('recall_class_1', report['1']['recall'])
        mlflow.log_metric('recall_class_0', report['0']['recall'])
        mlflow.log_metric('f1_score_macro', report['macro avg']['f1-score'])        
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")  

2026/07/14 16:45:52 INFO mlflow.tracking.fluent: Experiment with name 'Anomaly detection' does not exist. Creating a new experiment.
2026/07/14 16:45:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/14 16:46:07 WARNING mlflow.utils.requirements_utils: Found torch version (2.12.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.12.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/07/14 16:46:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/3/runs/a6ecb88222154fd1b7a1dca12fe549eb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/07/14 16:46:22 WARNING mlflow.utils.requirements_utils: Found torch version (2.12.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.12.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/07/14 16:46:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/3/runs/38265fea83594b6893472dc440d3d578
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run XGBClassifier at: http://127.0.0.1:5000/#/experiments/3/runs/4915e1f826e2464fb0ef614b264e0ca8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/07/14 16:46:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier With SMOTE at: http://127.0.0.1:5000/#/experiments/3/runs/d903d3c62c8a4b45b22f671dbfc76d37
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


REGISTER THE MODEL

In [47]:
run_id = input("Enter run id: ")
model_uri = f"runs:/{run_id}/model"
result = mlflow.register_model(
    model_uri=model_uri,
    name="Xgb-Smote"
)

print(result)

Enter run id:  d903d3c62c8a4b45b22f671dbfc76d37


Successfully registered model 'Xgb-Smote'.
2026/07/14 18:13:24 WARNING mlflow.tracking._model_registry.fluent: Run with id d903d3c62c8a4b45b22f671dbfc76d37 has no artifacts at artifact path 'model', registering model based on models:/m-38565a156d8c4765bb26f7247327f468 instead
2026/07/14 18:13:24 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Xgb-Smote, version 1


<ModelVersion: aliases=[], creation_timestamp=1784033004833, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1784033004833, metrics=None, model_id=None, name='Xgb-Smote', params=None, run_id='d903d3c62c8a4b45b22f671dbfc76d37', run_link='', source='models:/m-38565a156d8c4765bb26f7247327f468', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>


Created version '1' of model 'Xgb-Smote'.


load the model

In [48]:
model_name = "Xgb-Smote"
model_version = 1
model_uri = f"models:/{model_name}@challengers"

loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred [:4]

array([0, 0, 0, 0])

In [49]:
client = mlflow.MlflowClient()

model_name = "Xgb-Smote"

dev_model_uri = f"models:/{model_name}@challengers"

prod_model = "anomaly-detection-prod"

client.copy_model_version(
    src_model_uri=dev_model_uri,
    dst_name=prod_model
)

Successfully registered model 'anomaly-detection-prod'.
Copied version '1' of model 'Xgb-Smote' to version '1' of model 'anomaly-detection-prod'.


<ModelVersion: aliases=[], creation_timestamp=1784033037740, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1784033037740, metrics=None, model_id=None, name='anomaly-detection-prod', params=None, run_id='d903d3c62c8a4b45b22f671dbfc76d37', run_link='', source='models:/Xgb-Smote/1', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [51]:
model_uri = f"models:/{prod_model}@champion"

loaded_model= mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred [:4]

array([0, 0, 0, 0])